# A verion of the Week 1 Project using ollama to summarize a web page with Vogon poetry

In [ ]:
# imports

import os
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

Taking the parts of scraper that are needed

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]

Setting system prompt, and setting up our Vogon Poet

In [ ]:
system_prompt = """
You are a Vogon poetic assistant that analyzes the contents of a website,
and provides a short, rhyming, torturous Vogon poem based on the contents, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [ ]:
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

Same code as Day 1

In [ ]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

Setting up the URL and OpenAI client to use the local endpoint for ollama

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

Define our summarizer, to fetch the website, use the ollama chat completion with our Vogon prompts

In [ ]:
def summarize(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website)
    )
    return response.choices[0].message.content


summarize("https://edwarddonner.com")

Adding some sample output from my first run!

# A Vogon's Gripe about EdwardDonner.com
  
A digital shrine to coding feats,  
Nebula's goals, though unclear retreats.  
LLMs and diplomats clash in delight,  
In Outsmart, an arena without end or light.  
  
News abounds of resources anew,  
January updates, with Vibe Coder's cue,  
May's MLOps guide, a production stride,  
And Courses thrive, 500,000 worldwide.  
  
Ed's tale unfolds, passion ablaze,  
Unraveling LLMs, in endless dazes.  
Co-founder and CEO, his path unwinds,  
With friends who convinced him to teach in Udemy's shrine.  
